# Kiểm tra và chuẩn hóa nhãn PKU (YOLO)
**Mục tiêu:** chỉ đọc PKU gốc, kiểm tra ID 0–5 và `class cx cy w h` chuẩn hóa trong [0,1], chiều dương và bbox nằm trong ảnh. **Input:** `data/raw/PKU-Market-PCB(Data enhanced version)/{train,valid,test}/{images,labels}`, `Class.txt`. **Output khi chạy thành công:** `data/processed/labels/<source_split>/*.txt`, symlink ảnh `data/processed/images/<source_split>/*`. Đường dẫn song song `images/` ↔ `labels/` là yêu cầu để Ultralytics tìm nhãn; ảnh vẫn ở raw và chỉ được đọc qua symlink. Không dùng DeepPCB; không huấn luyện. Chạy notebook 08 → 09 → 10 → 11, dừng khi có lỗi.

In [1]:
from pathlib import Path
from collections import Counter
from IPython.display import display, Markdown
from PIL import Image
import math, os, re
import pandas as pd
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/PKU-Market-PCB(Data enhanced version)/Class.txt').is_file()), None)
if ROOT is None: raise FileNotFoundError('Chạy trong project PKU')
SOURCE = ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)'
SPLITS = ('train', 'valid', 'test')
EXPECTED = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']
classes = re.findall(r"['\"]([^'\"]+)['\"]", (SOURCE / 'Class.txt').read_text(encoding='utf-8-sig'))
if classes != EXPECTED: raise ValueError(f'Class.txt không đúng thứ tự 6 lớp PKU: {classes}')
display(pd.DataFrame({'class_id': range(len(classes)), 'class_name': classes}))
display(Markdown('Bảng trên lấy trực tiếp từ Class.txt và xác nhận thứ tự class ID; không đổi ID.'))


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,class_id,class_name
0,0,Missing_hole
1,1,Mouse_bite
2,2,Open_circuit
3,3,Short
4,4,Spur
5,5,Spurious_copper


Bảng trên lấy trực tiếp từ Class.txt và xác nhận thứ tự class ID; không đổi ID.

## Preflight toàn bộ file trước khi ghi, không bỏ qua annotation sai.

In [2]:
prepared, errors, statistics = [], [], []
exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
for source_split in SPLITS:
    images_dir, labels_dir = SOURCE / source_split / 'images', SOURCE / source_split / 'labels'
    images = [p for p in sorted(images_dir.iterdir()) if p.is_file() and p.suffix.lower() in exts]
    labels = [p for p in sorted(labels_dir.iterdir()) if p.is_file() and p.suffix.lower() == '.txt']
    image_by_stem = {p.stem: p for p in images}
    label_by_stem = {p.stem: p for p in labels}
    if len(image_by_stem) != len(images): errors.append(f'{source_split}: trùng stem ảnh khác extension')
    for stem in sorted(image_by_stem.keys() ^ label_by_stem.keys()): errors.append(f'{source_split}/{stem}: thiếu ảnh hoặc label')
    total_boxes = 0
    for stem in sorted(image_by_stem.keys() & label_by_stem.keys()):
        image, label = image_by_stem[stem], label_by_stem[stem]
        try:
            with Image.open(image) as im:
                width, height = im.size
                im.verify()
            lines = label.read_text(encoding='utf-8-sig').splitlines()
        except (OSError, UnicodeError, ValueError) as exc:
            errors.append(f'{label.relative_to(ROOT)}: không đọc được ({exc})')
            continue
        formatted = []
        for lineno, line in enumerate(lines, 1):
            if not line.strip(): continue  # YOLO cho phép label rỗng (ảnh âm tính)
            try:
                tokens = line.split()
                if len(tokens) != 5: raise ValueError('cần đúng 5 cột')
                cid, cx, cy, w, h = map(float, tokens)
                if not all(math.isfinite(v) for v in (cid, cx, cy, w, h)): raise ValueError('giá trị không hữu hạn')
                if not cid.is_integer() or not 0 <= int(cid) < len(classes): raise ValueError('class_id ngoài 0–5')
                if not all(0 <= v <= 1 for v in (cx, cy, w, h)): raise ValueError('tọa độ YOLO ngoài [0,1]')
                if w <= 0 or h <= 0 or cx - w/2 < -1e-9 or cy - h/2 < -1e-9 or cx + w/2 > 1+1e-9 or cy + h/2 > 1+1e-9: raise ValueError('bbox rỗng/vượt biên')
                formatted.append(' '.join([str(int(cid)), *(format(v, '.12g') for v in (cx, cy, w, h))]))
            except ValueError as exc: errors.append(f'{label.relative_to(ROOT)}:{lineno}: {exc}')
        total_boxes += len(formatted)
        # Kết thúc newline thống nhất, giữ thứ tự box gốc và độ chính xác 12 chữ số có nghĩa.
        prepared.append((source_split, image, label, ('\n'.join(formatted) + '\n') if formatted else ''))
    statistics.append({'source_split': source_split, 'images': len(images), 'labels': len(labels), 'valid_box_lines': total_boxes})
display(pd.DataFrame(statistics))
display(Markdown('Mỗi hàng là split **gốc**; phase 09 mới chia lại theo group. Ảnh/label phải ghép cùng stem, tổng box là số dòng hợp lệ.'))
if errors:
    display(pd.DataFrame({'issue': errors}))
    raise ValueError(f'FAIL: {len(errors)} lỗi PKU; chưa ghi data/processed. Sửa nguồn đầu vào có kiểm soát trước khi tiếp tục.')
display(Markdown(f'**PASS:** {len(prepared)} cặp hợp lệ, {sum(x["valid_box_lines"] for x in statistics)} box.'))


,source_split,images,labels,valid_box_lines
0,train,2772,2772,11812
1,valid,366,366,1307
2,test,366,366,1307


Mỗi hàng là split **gốc**; phase 09 mới chia lại theo group. Ảnh/label phải ghép cùng stem, tổng box là số dòng hợp lệ.

**PASS:** 3504 cặp hợp lệ, 14426 box.

In [3]:
for source_split, image, label, content in prepared:
    target_label = ROOT / 'data/processed/labels' / source_split / f'{label.stem}.txt'
    target_image = ROOT / 'data/processed/images' / source_split / image.name
    if target_label.exists() and target_label.read_text(encoding='utf-8') != content: raise FileExistsError(f'Label đã tồn tại và khác nội dung: {target_label}')
    if target_image.is_symlink() and target_image.resolve() != image.resolve(): raise FileExistsError(f'Symlink khác nguồn: {target_image}')
    if target_image.exists() and not target_image.is_symlink(): raise FileExistsError(f'Ảnh đích đã tồn tại: {target_image}')
    if target_image.is_symlink() and not target_image.exists(): raise FileNotFoundError(f'Symlink hỏng: {target_image}')
for source_split, image, label, content in prepared:
    target_label = ROOT / 'data/processed/labels' / source_split / f'{label.stem}.txt'
    target_image = ROOT / 'data/processed/images' / source_split / image.name
    target_label.parent.mkdir(parents=True, exist_ok=True)
    target_image.parent.mkdir(parents=True, exist_ok=True)
    if not target_label.exists(): target_label.write_text(content, encoding='utf-8')
    if not target_image.is_symlink(): target_image.symlink_to(os.path.relpath(image, target_image.parent))
display(Markdown(f'**Đạt:** đã chuẩn bị {len(prepared)} nhãn YOLO và symlink ảnh tương ứng trong `data/processed/`; raw không bị sửa. **Tiếp:** chạy 09 để chia theo group.'))


**Đạt:** đã chuẩn bị 3504 nhãn YOLO và symlink ảnh tương ứng trong `data/processed/`; raw không bị sửa. **Tiếp:** chạy 09 để chia theo group.

## Nhận xét / điều kiện kết thúc
Bảng class và thống kê split được **tính khi chạy**, không phải số liệu điền sẵn. PASS chỉ khi mọi ảnh/nhãn PKU ghép đúng và bbox hợp lệ; bất kỳ lỗi nào làm notebook dừng trước khi ghi output. Ảnh processed là symlink, phải giữ `data/raw` nguyên vị trí khi dùng Ultralytics. Không có DeepPCB trong outputs.